### Quant pricing library project

Focus on European financial instruments (vanilla, digital, barrier), with closed-form
Black-Scholes pricing, a Cox-Ross-Rubinstein binomial tree for American exercise, and
Monte Carlo pricing including two variance-reduction techniques.

In [1]:
from abc import ABC, abstractmethod

import numpy as np

from scipy.stats import norm

In [2]:
path = [100, 105, 110, 95, 130, 120]

## Financial instruments classes

In [3]:
class Product(ABC):
    """Abstract base class for any priceable financial instrument."""

    @abstractmethod
    def payoff(self):
        pass

In [4]:
class TerminalProduct(Product):
    """Base class for products whose payoff depends only on the terminal spot S_T."""

    def __init__(self, strike):
        self.strike = strike

In [5]:
class PathDependentProduct(Product):
    """Base class for products whose payoff depends on the whole simulated path."""

    def __init__(self, strike, barrier):
        self.strike = strike
        self.barrier = barrier

    @abstractmethod
    def payoff(self, path):
        pass

### Vanilla

**European call.** Gives the holder the right, but not the obligation, to
buy the underlying at the strike $K$ at maturity $T$. Exercised only if
$S_T > K$, so its payoff is:
$$\text{Payoff}_{\text{call}} = \max(S_T - K, 0)$$

**European put.** Symmetric right to *sell* at $K$: exercised only if
$S_T < K$.
$$\text{Payoff}_{\text{put}} = \max(K - S_T, 0)$$

Both depend only on the terminal spot $S_T$, not on the path taken to get
there — hence `TerminalProduct` as their common ancestor.

In [6]:
class Vanilla(TerminalProduct):
    """European vanilla option with a validated positive strike."""

    def __init__(self, strike):
        self.strike = strike

    @property
    def strike(self):
        return self.__strike

    @strike.setter
    def strike(self, value):
        if value <= 0:
            raise ValueError("Strike must be positive")
        self.__strike = value

    @abstractmethod
    def payoff(self):
        pass

In [7]:
class EuropeanCall(Vanilla):
    """European call: pays max(S_T - K, 0) at maturity."""

    def payoff(self, S_T):
        return np.maximum(S_T - self.strike, 0)

In [8]:
call = EuropeanCall(strike=100)
call.payoff(120)

np.int64(20)

In [9]:
class EuropeanPut(Vanilla):
    """European put: pays max(K - S_T, 0) at maturity."""

    def payoff(self, S_T):
        return np.maximum(self.strike - S_T, 0)

In [10]:
put = EuropeanPut(strike=100)
put.payoff(120)

np.int64(0)

### Path dependent

**Barrier option (up-and-in variant used here).** A vanilla call or put
that only comes into existence ("knocks in") if the underlying ever trades
at or above a barrier level $B$ during the option's life — otherwise it
expires worthless, even if it would have finished in the money. Because the
payoff depends on the whole path $(S_t)_{0\le t\le T}$, not just $S_T$, it
needs simulation (Monte Carlo) rather than the closed-form BSM formula.

**Up-and-in call.**
$$\text{Payoff} = \max(S_T - K, 0) \cdot \mathbb{1}\{\max_{0\le t\le T} S_t \ge B\}$$

**Up-and-in put.**
$$\text{Payoff} = \max(K - S_T, 0) \cdot \mathbb{1}\{\max_{0\le t\le T} S_t \ge B\}$$

where $\mathbb{1}\{\cdot\}$ is the indicator function (1 if the barrier was
touched, 0 otherwise). A knocked-in option is strictly cheaper than the
equivalent vanilla, since it pays off on a strict subset of the paths that
would make the vanilla pay off (cf. the structuring prep document, Q70-71:
KI + KO = vanilla).

In [11]:
class BarrierOption(PathDependentProduct):
    """Base class for barrier options (up-and-in / up-and-out variants)."""

    @abstractmethod
    def payoff(self, path):
        pass

In [12]:
class EuropeanUAICall(BarrierOption):
    """Up-and-in call: pays max(S_T - K, 0) only if the path ever reached the barrier."""

    def payoff(self, path):
        S_T = path[-1]

        if max(path) >= self.barrier:
            return max(S_T - self.strike, 0)
        else:
            return 0

In [13]:
UAI_call = EuropeanUAICall(strike=100, barrier=120)
UAI_call.payoff(path)

20

In [14]:
class EuropeanUAIPut(BarrierOption):
    """Up-and-in put: pays max(K - S_T, 0) only if the path ever reached the barrier."""

    def payoff(self, path):
        S_T = path[-1]

        if max(path) >= self.barrier:
            return max(self.strike - S_T, 0)
        else:
            return 0

In [15]:
UAI_put = EuropeanUAIPut(strike=100, barrier=120)
UAI_put.payoff(path)

0

### Digital

**Digital (cash-or-nothing) option.** Pays a fixed cash amount `money` if
the option finishes in the money, and exactly zero otherwise — unlike a
vanilla, the payout does not scale with how far in the money $S_T$ ends up.
Used standalone as a pure bet on direction, and as a building block inside
structured products (e.g. the coupon condition of an autocall is a digital,
cf. structuring prep document Q84, Q92).

**Digital call.**
$$\text{Payoff} = \text{money} \cdot \mathbb{1}\{S_T > K\}$$

**Digital put.**
$$\text{Payoff} = \text{money} \cdot \mathbb{1}\{S_T < K\}$$

In [16]:
class DigitalOption(Vanilla):
    """Cash-or-nothing digital option paying a fixed amount `money` if in the money."""

    def __init__(self, strike, money):
        self.strike = strike
        self.money = money

    @abstractmethod
    def payoff(self):
        pass

In [17]:
class EuropeanDigitalCall(DigitalOption):
    """Digital call: pays `money` if S_T > strike at maturity, else 0."""

    def payoff(self, S_T):
        return np.where(S_T > self.strike, self.money, 0)

In [18]:
dig_call = EuropeanDigitalCall(strike=100, money=200)
dig_call.payoff(120)

array(200)

In [19]:
class EuropeanDigitalPut(DigitalOption):
    """Digital put: pays `money` if S_T < strike at maturity, else 0."""

    def payoff(self, S_T):
        return np.where(S_T < self.strike, self.money, 0)

In [20]:
dig_put = EuropeanDigitalPut(strike=100, money=200)
# S_T = 120 > strike = 100, so the digital put must NOT pay off: expect 0.
dig_put.payoff(120)

array(0)

## Market data class

In [21]:
class MarketData:
    """Container for the market inputs (spot, risk-free rate, flat volatility)."""

    def __init__(self, spot, rate, vol):
        self.spot = spot
        self.rate = rate
        self.vol = vol

In [22]:
market = MarketData(spot=100, rate=0.04, vol=0.27)

## Pricing model base class

In [23]:
class PricingModel(ABC):
    """Abstract base class for any pricing engine."""

    @abstractmethod
    def pricing(self, product):
        pass

## Black-Scholes class

**Derivation sketch.** Under no-arbitrage, a delta-hedged portfolio
$\Pi = V - \Delta S$ is locally riskless (the $dW$ term cancels in $d\Pi$ by
Itô's lemma), so it must earn the risk-free rate: $d\Pi = r\Pi\,dt$. This
gives the Black-Scholes PDE:
$$\frac{\partial V}{\partial t} + rS\frac{\partial V}{\partial S} + \frac12\sigma^2S^2\frac{\partial^2 V}{\partial S^2} - rV = 0$$
By Feynman-Kac, the solution is $C_0 = e^{-rT}E^{\mathbb Q}[\max(S_T-K,0)]$,
where $\mathbb Q$ is the risk-neutral measure ($S_T$ lognormal under
$\mathbb Q$ with drift $r$, not the real-world drift $\mu$ — $\mu$ cancels
out of the hedging argument above, which is *why* pricing only needs $r$,
not $\mu$). Evaluating that expectation gives $C = SN(d_1) - Ke^{-rT}N(d_2)$.

**A key identity, used repeatedly below:** $S\phi(d_1) = Ke^{-rT}\phi(d_2)$
(follows from $d_1^2-d_2^2 = 2\ln(S/K)+2rT$). Every Greek derivation below
leans on this identity to make most cross-terms cancel.

**Delta.** Differentiating $C=SN(d_1)-Ke^{-rT}N(d_2)$ w.r.t. $S$ gives
$N(d_1) + S\phi(d_1)\frac{\partial d_1}{\partial S} - Ke^{-rT}\phi(d_2)\frac{\partial d_2}{\partial S}$.
Since $d_2=d_1-\sigma\sqrt T$, $\frac{\partial d_1}{\partial S}=\frac{\partial d_2}{\partial S}=\frac{1}{S\sigma\sqrt T}$,
so the last two terms become identical (via the identity above) and cancel,
leaving $\boxed{\Delta = N(d_1)}$. For a put, $\Delta=N(d_1)-1$ (put-call
parity differentiated w.r.t. $S$).

**Gamma.** A direct chain-rule consequence of $\Delta=N(d_1)$:
$\Gamma=\frac{\partial\Delta}{\partial S}=\phi(d_1)\frac{\partial d_1}{\partial S}=\frac{\phi(d_1)}{S\sigma\sqrt T}$.
Always positive, and maximal exactly where $\phi(d_1)$ is maximal, i.e. at
$d_1=0$ (same spot level as the vega peak below).

**Vega.** $\frac{\partial C}{\partial\sigma}=S\phi(d_1)\left[\frac{\partial d_1}{\partial\sigma}-\frac{\partial d_2}{\partial\sigma}\right]$
(identity again), and since $d_2=d_1-\sigma\sqrt T$ that bracket is exactly
$\sqrt T$, giving $\mathcal V = S\phi(d_1)\sqrt T$. Because $\phi$ is the
standard normal density, vega is bell-shaped in $S$ and peaks at $d_1=0$.
Solving $d_1=0$ for $S$:
$$\ln(S/K) + \left(r+\tfrac12\sigma^2\right)T = 0 \;\Rightarrow\; S = Ke^{-(r+\frac12\sigma^2)T}$$
So the true vega peak is spot-at-the-money adjusted by a drift term, **not
exactly $S=K$** — the common shorthand "vega peaks ATM" only holds well for
short maturities / low rates, where that correction factor is close to 1.

In [24]:
class BlackScholes(PricingModel):
    """
    Closed-form Black-Scholes-Merton pricer and Greeks (no dividend yield).

    Supports European vanilla calls/puts (pricing, delta, gamma, vega) and
    cash-or-nothing digital calls/puts (pricing only). Raises
    NotImplementedError for any other product type rather than failing silently.
    See the markdown cell above for the derivations.
    """

    def __init__(self, market, maturity):
        self.market = market
        self.maturity = maturity

    def _d1(self, product):
        K = product.strike
        return (
            np.log(self.market.spot / K)
            + (self.market.rate + 0.5 * self.market.vol**2) * self.maturity
        ) / (self.market.vol * np.sqrt(self.maturity))

    def _d2(self, product):
        return self._d1(product) - self.market.vol * np.sqrt(self.maturity)

    def pricing(self, product):
        """Closed-form BSM price. Cross-check: C - P = S - K*exp(-r*T) (put-call parity)."""
        K = product.strike
        d1 = self._d1(product)
        d2 = self._d2(product)
        spot = self.market.spot
        rate = self.market.rate

        if isinstance(product, EuropeanDigitalCall):
            return product.money * np.exp(-rate * self.maturity) * norm.cdf(d2)

        elif isinstance(product, EuropeanDigitalPut):
            return product.money * np.exp(-rate * self.maturity) * norm.cdf(-d2)

        elif isinstance(product, EuropeanCall):
            return spot * norm.cdf(d1) - K * np.exp(-rate * self.maturity) * norm.cdf(d2)

        elif isinstance(product, EuropeanPut):
            return K * np.exp(-rate * self.maturity) * norm.cdf(-d2) - spot * norm.cdf(-d1)

        else:
            raise NotImplementedError(
                f"BlackScholes.pricing is not implemented for {type(product).__name__}"
            )

    def delta(self, product):
        """Analytical BSM delta: N(d1) for a call, N(d1) - 1 for a put."""
        if not isinstance(product, (EuropeanCall, EuropeanPut)):
            raise NotImplementedError(
                f"BlackScholes.delta is not implemented for {type(product).__name__}"
            )
        d1 = self._d1(product)
        if isinstance(product, EuropeanCall):
            return norm.cdf(d1)
        else:
            return norm.cdf(d1) - 1

    def gamma(self, product):
        """Analytical BSM gamma (identical formula for call and put)."""
        if not isinstance(product, (EuropeanCall, EuropeanPut)):
            raise NotImplementedError(
                f"BlackScholes.gamma is not implemented for {type(product).__name__}"
            )
        d1 = self._d1(product)
        return norm.pdf(d1) / (self.market.spot * self.market.vol * np.sqrt(self.maturity))

    def vega(self, product):
        """Analytical BSM vega (identical formula for call and put)."""
        if not isinstance(product, (EuropeanCall, EuropeanPut)):
            raise NotImplementedError(
                f"BlackScholes.vega is not implemented for {type(product).__name__}"
            )
        d1 = self._d1(product)
        return self.market.spot * norm.pdf(d1) * np.sqrt(self.maturity)

In [25]:
bs = BlackScholes(market, maturity=1)
print("The price of the put is:", round(bs.pricing(put), 3))

The price of the put is: 8.682


In [26]:
bs = BlackScholes(market, maturity=1)
print("The price of the digital call is:", round(bs.pricing(dig_call), 3))

The price of the digital call is: 97.087


## Binomial tree (Cox-Ross-Rubinstein) — European and American vanilla options

In [27]:
class BinomialTree(PricingModel):
    """
    Cox-Ross-Rubinstein recombining binomial tree pricer for vanilla options.

    Supports both European exercise (should converge to the Black-Scholes price
    as n_steps grows) and American exercise (early-exercise check at every node).
    This is the main differentiator versus the terminal-payoff-only pricers above:
    it is the only engine in this library that can value early-exercise features.
    """

    def __init__(self, spot, rate, vol, maturity, n_steps, american=False):
        self.spot = spot
        self.rate = rate
        self.vol = vol
        self.maturity = maturity
        self.n_steps = n_steps
        self.american = american

    def pricing(self, product):
        dt = self.maturity / self.n_steps
        u = np.exp(self.vol * np.sqrt(dt))
        d = 1 / u
        p = (np.exp(self.rate * dt) - d) / (u - d)
        discount = np.exp(-self.rate * dt)

        # Terminal spot prices at every node of the final layer (vectorised).
        j = np.arange(self.n_steps + 1)
        S_T = self.spot * (u**j) * (d ** (self.n_steps - j))
        values = product.payoff(S_T)

        # Backward induction through the tree.
        for step in range(self.n_steps - 1, -1, -1):
            values = discount * (p * values[1:] + (1 - p) * values[:-1])

            if self.american:
                j = np.arange(step + 1)
                S_t = self.spot * (u**j) * (d ** (step - j))
                intrinsic_value = product.payoff(S_t)
                values = np.maximum(values, intrinsic_value)

        return values[0]

In [28]:
# Convergence check: as n_steps grows, the European binomial price should approach
# the Black-Scholes closed-form price for the same call.
for n_steps in (50, 200, 1000):
    tree = BinomialTree(spot=100, rate=0.04, vol=0.27, maturity=1, n_steps=n_steps, american=False)
    print(f"n_steps={n_steps:>4} | European binomial call price = {tree.pricing(call):.4f}")

print(f"Black-Scholes call price            = {bs.pricing(call):.4f}")

n_steps=  50 | European binomial call price = 12.5504
n_steps= 200 | European binomial call price = 12.5901
n_steps=1000 | European binomial call price = 12.6008
Black-Scholes call price            = 12.6034


In [29]:
# American vs European put: the American price should be >= the European price,
# since early exercise is only ever exercised when it adds value.
euro_tree = BinomialTree(spot=100, rate=0.04, vol=0.27, maturity=1, n_steps=500, american=False)
amer_tree = BinomialTree(spot=100, rate=0.04, vol=0.27, maturity=1, n_steps=500, american=True)

print(f"European put (tree) : {euro_tree.pricing(put):.4f}")
print(f"American put (tree) : {amer_tree.pricing(put):.4f}")

European put (tree) : 8.6770
American put (tree) : 9.0764


## Monte Carlo — terminal-payoff products, with variance reduction

In [30]:
class MonteCarloTerminal(PricingModel):
    """
    Monte Carlo pricer for European (terminal-payoff) options under GBM.

    Three pricing modes are available:
    - plain simulation (`pricing`)
    - antithetic variates (`pricing(..., antithetic=True)`)
    - a control variate based on the terminal spot S_T itself, whose risk-neutral
      expectation is known analytically (`pricing_control_variate`)
    """

    def __init__(self, spot, rate, vol, maturity, n_simulations):
        self.spot = spot
        self.rate = rate
        self.vol = vol
        self.maturity = maturity
        self.n_simulations = n_simulations

    def _simulate_terminal_spot(self, Z):
        """Simulate terminal spot prices from standard normal draws Z (vectorised)."""
        return self.spot * np.exp(
            (self.rate - 0.5 * self.vol**2) * self.maturity
            + self.vol * np.sqrt(self.maturity) * Z
        )

    def pricing(self, product, antithetic=False, return_stderr=False):
        """
        Price a European option by plain or antithetic Monte Carlo.

        If return_stderr is True, also returns the Monte Carlo standard error
        of the price estimate, which can be used to build a confidence interval
        (e.g. price +/- 1.96 * stderr for an approximate 95% CI).
        """
        if antithetic:
            n_half = self.n_simulations // 2
            Z = np.random.normal(size=n_half)
            S_T_up = self._simulate_terminal_spot(Z)
            S_T_down = self._simulate_terminal_spot(-Z)
            discounted_payoffs = np.exp(-self.rate * self.maturity) * 0.5 * (
                product.payoff(S_T_up) + product.payoff(S_T_down)
            )
        else:
            Z = np.random.normal(size=self.n_simulations)
            S_T = self._simulate_terminal_spot(Z)
            discounted_payoffs = np.exp(-self.rate * self.maturity) * product.payoff(S_T)

        price = np.mean(discounted_payoffs)

        if return_stderr:
            stderr = np.std(discounted_payoffs, ddof=1) / np.sqrt(len(discounted_payoffs))
            return price, stderr
        return price

    def pricing_control_variate(self, product):
        """
        Price by Monte Carlo using the terminal spot S_T as a control variate.

        E_Q[S_T] = spot * exp(rate * maturity) is known in closed form, so we can
        subtract off b * (S_T - E_Q[S_T]) from each simulated discounted payoff,
        where b is chosen to minimise the variance of the adjusted estimator:
        b* = Cov(payoff, S_T) / Var(S_T).
        """
        Z = np.random.normal(size=self.n_simulations)
        S_T = self._simulate_terminal_spot(Z)
        discount = np.exp(-self.rate * self.maturity)

        payoffs = product.payoff(S_T)
        control = S_T
        control_expectation = self.spot * np.exp(self.rate * self.maturity)

        b = np.cov(payoffs, control)[0, 1] / np.var(control)

        adjusted_payoffs = discount * (payoffs - b * (control - control_expectation))
        price = np.mean(adjusted_payoffs)
        stderr = np.std(adjusted_payoffs, ddof=1) / np.sqrt(self.n_simulations)

        return price, stderr

In [31]:
bs = BlackScholes(market, maturity=1)
mc = MonteCarloTerminal(spot=market.spot, rate=market.rate, vol=market.vol, maturity=1, n_simulations=1_000_000)

price_bs = bs.pricing(call)
price_mc, stderr_mc = mc.pricing(call, return_stderr=True)
price_mc_anti, stderr_anti = mc.pricing(call, antithetic=True, return_stderr=True)
price_mc_cv, stderr_cv = mc.pricing_control_variate(call)

ci95 = 1.96 * stderr_mc

print(f"Black-Scholes            : {price_bs:.4f}")
print(f"Monte Carlo (plain)      : {price_mc:.4f}  (stderr={stderr_mc:.4f}, 95% CI +/- {ci95:.4f})")
print(f"Monte Carlo (antithetic) : {price_mc_anti:.4f}  (stderr={stderr_anti:.4f})")
print(f"Monte Carlo (control var): {price_mc_cv:.4f}  (stderr={stderr_cv:.4f})")
print(f"|BS - MC plain|          : {abs(price_bs - price_mc):.4f}")
print(f"Variance reduction factor, antithetic : {(stderr_mc / stderr_anti) ** 2:.2f}x")
print(f"Variance reduction factor, control var: {(stderr_mc / stderr_cv) ** 2:.2f}x")

Black-Scholes            : 12.6034
Monte Carlo (plain)      : 12.6095  (stderr=0.0198, 95% CI +/- 0.0388)
Monte Carlo (antithetic) : 12.5960  (stderr=0.0152)
Monte Carlo (control var): 12.6137  (stderr=0.0077)
|BS - MC plain|          : 0.0061
Variance reduction factor, antithetic : 1.68x
Variance reduction factor, control var: 6.55x


In [32]:
dig_call_mc = EuropeanDigitalCall(strike=100, money=500)
mc_dig = MonteCarloTerminal(spot=market.spot, rate=market.rate, vol=market.vol, maturity=1, n_simulations=100_000)
price_dig = mc_dig.pricing(dig_call_mc)
print("Price of digital call (MC):", price_dig)

Price of digital call (MC): 243.6225741386588


## Monte Carlo — path-dependent products (barrier options)

In [33]:
class MonteCarloPathDependent(PricingModel):
    """
    Monte Carlo pricer for path-dependent options (e.g. barrier options) under GBM.

    Paths are generated in a single vectorised (n_simulations, n_steps + 1) array
    rather than with nested Python loops, which is the dominant cost of naive
    path-dependent Monte Carlo and is orders of magnitude faster once vectorised.
    """

    def __init__(self, spot, rate, vol, maturity, n_simulations, n_steps, seed=None):
        self.spot = spot
        self.rate = rate
        self.vol = vol
        self.maturity = maturity
        self.n_simulations = n_simulations
        self.n_steps = n_steps
        self.seed = seed

    def generate_paths(self):
        """Generate all simulated paths at once as an (n_simulations, n_steps + 1) array."""
        dt = self.maturity / self.n_steps
        rng = np.random.default_rng(self.seed)
        Z = rng.normal(size=(self.n_simulations, self.n_steps))

        log_increments = (self.rate - 0.5 * self.vol**2) * dt + self.vol * np.sqrt(dt) * Z
        log_paths = np.cumsum(log_increments, axis=1)

        paths = self.spot * np.exp(log_paths)
        paths = np.hstack([np.full((self.n_simulations, 1), self.spot), paths])
        return paths

    def pricing(self, product):
        """
        Price a path-dependent product by averaging its discounted payoff over
        all simulated paths. Path generation is vectorised; the payoff loop below
        remains O(n_simulations) (not O(n_simulations x n_steps)), since the
        product.payoff() interface takes one path at a time to stay compatible
        with the single-path usage elsewhere in this notebook.
        """
        paths = self.generate_paths()
        payoffs = np.array([product.payoff(path) for path in paths])
        return np.exp(-self.rate * self.maturity) * np.mean(payoffs)

    def delta(self, product, h=1.0):
        """
        Estimate delta by central finite difference, using the SAME random draws
        (common random numbers, via a shared seed) for the up- and down-bumped spot.

        Without common random numbers, the up and down price estimates come from
        independent simulations, and the Monte Carlo noise of each estimate swamps
        the true delta signal. Sharing the seed removes that source of noise and
        isolates the effect of the spot bump.
        """
        seed = self.seed if self.seed is not None else np.random.randint(0, 2**31 - 1)

        mc_up = MonteCarloPathDependent(
            spot=self.spot + h, rate=self.rate, vol=self.vol,
            maturity=self.maturity, n_simulations=self.n_simulations,
            n_steps=self.n_steps, seed=seed,
        )
        mc_down = MonteCarloPathDependent(
            spot=self.spot - h, rate=self.rate, vol=self.vol,
            maturity=self.maturity, n_simulations=self.n_simulations,
            n_steps=self.n_steps, seed=seed,
        )
        return (mc_up.pricing(product) - mc_down.pricing(product)) / (2 * h)

In [34]:
uai_call = EuropeanUAICall(strike=100, barrier=120)
uai_put = EuropeanUAIPut(strike=100, barrier=120)

mc_path = MonteCarloPathDependent(spot=100, rate=0.05, vol=0.20, maturity=1, n_simulations=100_000, n_steps=252, seed=42)

print("Up-and-in call price :", mc_path.pricing(uai_call))
print("Up-and-in call delta :", mc_path.delta(uai_call))
print("Up-and-in put price  :", mc_path.pricing(uai_put))

Up-and-in call price : 9.163531047698118


Up-and-in call delta : 0.6575662369174813


Up-and-in put price  : 0.1728337968123544


## Validation & tests

Lightweight assertion-based checks (not yet migrated to a standalone pytest
module — see "Known limitations" below). Each check prints a confirmation on
success and raises an AssertionError with a descriptive message on failure.

In [35]:
def check(label, condition):
    assert condition, f"FAILED: {label}"
    print(f"OK   : {label}")


# 1. Put-call parity (model-free): C - P == S - K * exp(-rT)
lhs = bs.pricing(call) - bs.pricing(put)
rhs = market.spot - call.strike * np.exp(-market.rate * 1)
check("Put-call parity (BSM)", abs(lhs - rhs) < 1e-8)

# 2. Digital payoff sanity checks (this is exactly the bug we fixed: a digital put
#    struck at 100 must NOT pay off when S_T = 120).
check("Digital call pays off when S_T > strike", dig_call.payoff(120) == 200)
check("Digital put does NOT pay off when S_T > strike", dig_put.payoff(120) == 0)
check("Digital put pays off when S_T < strike", EuropeanDigitalPut(strike=100, money=200).payoff(80) == 200)

# 3. BSM vs plain Monte Carlo: should agree within ~4 standard errors (generous
#    tolerance to keep the test robust to random seed variation).
check(
    "BSM vs Monte Carlo (plain) agree within 4 stderr",
    abs(price_bs - price_mc) < 4 * stderr_mc,
)

# 4. Variance reduction actually reduces the standard error.
check("Antithetic variates reduce standard error", stderr_anti < stderr_mc)
check("Control variate reduces standard error", stderr_cv < stderr_mc)

# 5. Binomial tree converges to Black-Scholes for European exercise.
tree_fine = BinomialTree(spot=100, rate=0.04, vol=0.27, maturity=1, n_steps=1000, american=False)
check(
    "Binomial tree (1000 steps, European) converges to Black-Scholes",
    abs(tree_fine.pricing(call) - bs.pricing(call)) < 0.05,
)

# 6. American put must be worth at least as much as the European put (early
#    exercise is optional, so it can only add value, never subtract it).
check("American put >= European put", amer_tree.pricing(put) >= euro_tree.pricing(put) - 1e-8)

# 7. Delta cross-check: BSM analytical delta vs a bump-and-reprice estimate using
#    the SAME Monte Carlo draws for the base, up and down scenarios (common random
#    numbers), which should be far more precise than independent simulations.
np.random.seed(123)
h = 0.01
Z_shared = np.random.normal(size=2_000_000)


def mc_price_with_Z(spot, Z):
    S_T = spot * np.exp((market.rate - 0.5 * market.vol**2) * 1 + market.vol * np.sqrt(1) * Z)
    return np.exp(-market.rate * 1) * np.mean(call.payoff(S_T))


delta_mc_crn = (mc_price_with_Z(market.spot + h, Z_shared) - mc_price_with_Z(market.spot - h, Z_shared)) / (2 * h)
delta_bs = bs.delta(call)

check(
    "Delta: BSM analytical vs Monte Carlo (common random numbers) agree within 0.01",
    abs(delta_mc_crn - delta_bs) < 0.01,
)

print("\nAll checks passed.")

OK   : Put-call parity (BSM)
OK   : Digital call pays off when S_T > strike
OK   : Digital put does NOT pay off when S_T > strike
OK   : Digital put pays off when S_T < strike
OK   : BSM vs Monte Carlo (plain) agree within 4 stderr
OK   : Antithetic variates reduce standard error
OK   : Control variate reduces standard error
OK   : Binomial tree (1000 steps, European) converges to Black-Scholes
OK   : American put >= European put
OK   : Delta: BSM analytical vs Monte Carlo (common random numbers) agree within 0.01

All checks passed.


## Known limitations

- No dividend yield: all pricing assumes q = 0. Extending to a continuous
  dividend yield would only require adding a `spot * exp(-q * T)` discount
  factor consistently across the BSM formulas and the drift term in the
  simulations.
- Analytical Greeks (delta/gamma/vega) are only implemented for vanilla
  European calls and puts, not for digital or barrier options; calling them
  on an unsupported product raises `NotImplementedError` rather than
  returning a silently wrong number.
- In `MonteCarloPathDependent.pricing`, path *generation* is fully vectorised,
  but the payoff evaluation loop is still O(n_simulations) in pure Python,
  since `payoff()` takes one path at a time to stay compatible with the
  single-path usage earlier in this notebook. Vectorising payoff evaluation
  too would require redesigning that interface to accept a batch of paths.
- The binomial tree currently only prices vanilla (non-path-dependent)
  products; extending it to barrier options would require checking the
  barrier condition at each node during the backward induction.
- Volatility is assumed flat (no smile/skew) throughout; every pricer here
  uses a single constant `vol` from `MarketData`.
- Tests are inline assertions in this notebook rather than a standalone
  `pytest` module — a natural next step would be to split this into a proper
  `pricing_library/` package with `tests/test_*.py` files.